### bpz_theory
---------------

This short notebook explains the theory behind the correction we do for b_p(z) with a toy model.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
import importlib
import healpy as hp
import astropy.units as u
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.cm as cm

import src.statistics.corrfiles as cf
import src.statistics.cosmotools as ct
import src.analysis.plots as plots
import src.analysis.maps as maps
import pyccl as ccl

from pathlib import Path

importlib.reload(plots)

PAPER_FIGURES_ROOT = Path('/global/cfs/projectdirs/desi/users/jchdj/desi-y3-hsc/paper/figures/')
moc_list = sorted([
    Path(
        '/global/cfs/projectdirs/desi/users/jchdj/desi-y3-hsc/data/mocs/', 
        f'hsc_moc{i+1}.fits'
    )
    for i in range(0, 4)
])
cmap_hsc = plt.get_cmap('plasma')
cmap_desi = plt.get_cmap('viridis')

# plot infrastructure
pm = plots.PlotManager(root=PAPER_FIGURES_ROOT, overwrite=True)

In [ ]:
### First, we assume some $n_p(z)$ distributions that are similar to the HSC distributions. 
# For this, we will assume gaussians of width 0.3 (photometric redshift spacing) and of mean the center of the hsc bins.

hsc_edges = np.linspace(0.3, 1.8, 0.3)
hsc_means = (hsc_edges[1:] + hsc_edges[:-1])/2
width = 0.3
zv = np.linspace(0, 2.5, 1000)
nzl = [np.exp(-0.5*((zv - mean)/width)**2) for mean in hsc_means]
nzl = [n/np.trapz(n, zv) for n in nzl]

## we will also assume a generic bias model bp(z) : 1/D(z) and see if we can recover it theoretically.
def bias_p(z):
    gf = ccl.background.growth_factor(ct.COSMO_ccl, a=1/(1+z))
    return 1/gf

In [ ]:
## now let's show the nz that we theoretically measure with our correction
n_meas = nzl * bias_p(z=zv)